In [1]:
from PIL import Image
import numpy as np
import os
import sys
import time

sys.path.append("/home/alberto/UnReflectAnything/")
from utilities.visualization import rgb, panelize
import torch
from dataset.scrream import SCRREAM
from rich import print
%load_ext autoreload
%autoreload 2


dataset = SCRREAM(
        root_dir="/datasets/SCRREAM/",
        rho_s=0.6,
        eps=1e-8
    )
    
# Create dataloader
dataloader = torch.utils.data.DataLoader(dataset, batch_size=1, shuffle=True)

# Test loading a batch
for batch in dataloader:
    batch = {k: v.cuda() if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
    break

# rgb(batch["rgb"][0],colormap="hsv")


In [2]:
from models import DINOv3, DINOv3toDPTRGB

dinov3_config = {
    'model_name': "facebook/dinov3-vits16-pretrain-lvd1689m",
    'image_size': 896,  # or any size divisible by 16
    'freeze_backbone': True,
    'return_selected_layers': [2, 5, 8, 11],  # DPT extraction points
    'return_as_feature_maps': True  # Need tokens for reassembly
}
dinov3_model = DINOv3(dinov3_config)

# Create the complete model with RGB decoder
model = DINOv3toDPTRGB(
    dinov3_model=dinov3_model,
    decoder_config={
        'use_bn': True,  # Use batch norm for training stability
        'readout_type': 'project'  # or 'project' for global context
    },
    selected_layers=[2, 5, 8, 11]  # Standard DPT layers
).cuda()

In [3]:
batch["rgb"].shape

torch.Size([1, 3, 874, 1132])

In [4]:
model.dinov3.patch_size*28

448

In [5]:
from models import DINOv3, DPTRGBDecoder, RGBPOLDecomposer, POLViTEncoder

dinov3_cfg = {
    "model_name": "facebook/dinov3-vits16-pretrain-lvd1689m",
    "image_size": 448,
    "freeze_backbone": True,
    "return_last_hidden_state": True,
    "return_as_feature_maps": False,  # Need tokens for cross-attention
}

pol_enc_cfg = {
    "in_ch": 3,
    "embed_dim": 384,  # ViT-S dimension
    "depth": 4,
    "n_heads": 12,
    "patch_size": 16,
}
decS_cfg = {
    "use_bn": True,
    "readout_type": "project",
    "feature_dim": 384,
    "output_image_size": (448, 448),
}
decD_cfg = {
    "use_bn": True,
    "readout_type": "project",
    "feature_dim": 384,
    "output_image_size": (448, 448),
}
decH_cfg = {
    "use_bn": True,
    "readout_type": "project",
    "feature_dim": 384,
    "output_image_size": (448, 448),
}
pol_enc = POLViTEncoder(pol_enc_cfg)
dinov3 = DINOv3(dinov3_cfg)
decS = DPTRGBDecoder(decS_cfg)
decD = DPTRGBDecoder(decD_cfg)
decH = DPTRGBDecoder(decH_cfg)

model = RGBPOLDecomposer(
    dinov3=dinov3,
    pol_encoder=pol_enc,
    pol_preprocess=None,  # Use default
    pol_cross_attn=None,  # Use default
    spec_decoder=decS,
    diffuse_decoder=decD,
    highlight_decoder=decH,
).cuda()



In [6]:
batch["rgb"].shape

torch.Size([1, 3, 874, 1132])

In [14]:
start = time.time()
decomposition = model(batch) 
end = time.time()
print("rgb:",batch["rgb"].shape)
print("AoLp:",batch["AoP"].shape)
print("DoLp:",batch["DoP"].shape)
def print_tensor_shapes(data, prefix=""):
    """Recursively print tensor names and shapes from nested dicts/lists"""
    if isinstance(data, torch.Tensor):
        print(f"{prefix}: {data.shape}")
    elif isinstance(data, dict):
        for key, value in data.items():
            new_prefix = f"{prefix}.{key}" if prefix else key
            print_tensor_shapes(value, new_prefix)
    elif isinstance(data, (list, tuple)):
        for i, item in enumerate(data):
            new_prefix = f"{prefix}[{i}]" if prefix else f"[{i}]"
            print_tensor_shapes(item, new_prefix)
    else:
        print(f"{prefix}: {type(data)} (not a tensor)")

print("Tensor Outputs:")
print_tensor_shapes(decomposition)
print(f"Time taken: {end - start} seconds")

rgb:
torch.Size([1, 3, 874, 1132])

AoLp:
torch.Size([1, 1, 874, 1132])

DoLp:
torch.Size([1, 1, 874, 1132])

Tensor Outputs:

specular: torch.Size([1, 3, 448, 448])

...

highlight: torch.Size([1, 3, 448, 448])

tokens.rgb: torch.Size([1, 789, 384])

tokens.pol: torch.Size([1, 3780, 384])

tokens.cross: torch.Size([1, 789, 384])

Time taken: 0.06278872489929199 seconds

In [8]:
from torchinfo import summary
summary(model)

Layer (type:depth-idx)                                            Param #
RGBPOLDecomposer                                                  --
├─DINOv3: 1-1                                                     --
│    └─DINOv3ViTModel: 2-1                                        --
│    │    └─DINOv3ViTEmbeddings: 3-1                              (297,600)
│    │    └─DINOv3ViTRopePositionEmbedding: 3-2                   --
│    │    └─ModuleList: 3-3                                       (21,298,176)
│    │    └─LayerNorm: 3-4                                        (768)
├─PolarizationPreprocess: 1-2                                     --
├─POLViTEncoder: 1-3                                              1,451,520
│    └─PatchEmbed: 2-2                                            --
│    │    └─Conv2d: 3-5                                           295,296
│    └─ModuleList: 2-3                                            --
│    │    └─TransformerBlock: 3-6                                 